# Adaptation of MultiCare Notebook. For Data Extraction (Phase 0)

**Install the `multiversity` library, import the `MedicalDatasetCreator` class, and instantiate it.**

In [ ]:
!pip install multiversity pandas

In [3]:
from multiversity.multicare_dataset import MedicalDatasetCreator

In [ ]:
# By default, the name of big dataset is 'medical_datasets'

mdc = MedicalDatasetCreator(directory = 'medical_datasets')

**Configure filter for best extraction quality**

In [20]:
def load_filter_config(keywords):
    return [
        {'field': 'keywords', 'string_list': keywords, 'match_type': 'partial_match'},

        {'field': 'label', 'string_list': ['chart', 'animal'], 'operator': 'none'},    # Exclude charts and animals
    ]

**Extract disease using configured filter**

In [15]:
def extract_subdataset(disease_name : str, disease_keywords, dataset_type='multimodal'):

    mdc.create_dataset(dataset_name=disease_name, filter_list=load_filter_config(keywords=disease_keywords), dataset_type=dataset_type)

In [16]:
def format_disease_keywords(disease_name, disease_keywords):
    # Convert keywords (even single) to a list
    if isinstance(disease_keywords, str):
        disease_keywords = [disease_keywords]

    return {'disease_name': disease_name, 'disease_keywords': disease_keywords}

In [22]:
import pandas as pd

DISEASES = list()

df = pd.read_csv('keywords.csv')    # Read keywords from a .csv file

for keyword in df['keyword']:
    DISEASES.append(format_disease_keywords(keyword, keyword))

In [ ]:
for disease in DISEASES:
    disease_name = disease['disease_name']
    disease_keywords = disease['disease_keywords']

    # Create subdataset
    extract_subdataset(disease_name, disease_keywords)

**Put individual subdatasets in one dataset**

In [ ]:
import os
import shutil
import time

# Generate a unique number for the folder to prevent overwrite collisions
dataset_number = int(time.time())
extracted_dir = f"extracted_dataset_{dataset_number}"

# cd into medical_datasets
os.chdir("medical_datasets")

# mkdir "extracted_dataset_<number>"
os.makedirs(extracted_dir, exist_ok=True)


# Move subdatasets into extracted_dir (exclude itself)
for item in os.listdir("."):
    if os.path.isdir(item) and item not in ["whole_multicare_dataset", extracted_dir]:
        target_path = os.path.join(extracted_dir, item)
        shutil.move(item, target_path)

# Move "extracted_dataset_<number>" outside of medical_datasets
shutil.move(extracted_dir, os.path.join("..", extracted_dir))


# cd outside as well
os.chdir("..")

print(f"Successfully organized datasets into: {extracted_dir}")